## 1 | Datensatz kurz vorgestellt**Quelle:** Hotel Booking Demand Dataset (Kaggle / Antonio, Almeida & Nunes, 2019).Der Datensatz enthaelt ~119.000 Hotelbuchungen (Resort- und Stadthotel) mit 31 Merkmalenwie Vorlaufzeit, Gaestezahl, Zimmertyp, Marktsegment und Sonderwuensche.**Zielvariable:** `reservation_status` mit 3 Klassen:- **Check-Out** – Gast hat eingecheckt und ausgecheckt- **Canceled** – Buchung wurde storniert- **No-Show** – Gast ist nicht erschienen**Achtung Data Leakage:** Die Spalten `is_canceled` und `reservation_status_date` werdenentfernt, da sie direkt aus dem Target abgeleitet sind.

In [ ]:
import sys; sys.path.insert(0, '..')import yamlfrom pathlib import Pathimport pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as snsimport warnings; warnings.filterwarnings('ignore')from src.data_loading import set_seedsfrom src.plotting import setup_plot_stylecfg = yaml.safe_load(open(Path('../configs/default.yaml')))set_seeds(cfg['random_seed'])setup_plot_style(cfg)

In [ ]:
# Daten ladenhb_cfg = cfg['hotel_bookings']raw_path = Path('..') / cfg['paths']['raw_data']df = pd.read_csv(raw_path / hb_cfg['file'])print(f'Shape: {df.shape}')df.head()

In [ ]:
df.info()

## 2 | Exploratory Data Analysis (EDA)

In [ ]:
# Klassenverteilung der Zielvariabletarget = hb_cfg['target']print(df[target].value_counts())print(f'\nProzentual:\n{df[target].value_counts(normalize=True).round(4) * 100}')plt.figure(figsize=(8, 5))ax = df[target].value_counts().plot.bar(color=['#2ecc71', '#e74c3c', '#f39c12'], edgecolor='black')plt.title('Verteilung von reservation_status', fontsize=14)plt.xlabel('Status')plt.ylabel('Anzahl')plt.xticks(rotation=0)for p in ax.patches:    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),                ha='center', va='bottom', fontsize=11)plt.tight_layout()plt.show()

Die Klassen sind **stark unbalanciert**: Check-Out und Canceled dominieren, No-Show macht nur einen sehr kleinen Anteil aus. Das muss beim Modelltraining beruecksichtigt werden.

In [ ]:
# Korrelation numerischer Featuresnum_cols = df.select_dtypes(include=[np.number]).columns.tolist()# Leakage-Spalten ausschliessennum_cols = [c for c in num_cols if c not in hb_cfg['drop_columns']]plt.figure(figsize=(14, 10))corr = df[num_cols].corr()mask = np.triu(np.ones_like(corr, dtype=bool))sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})plt.title('Korrelationsmatrix numerischer Features', fontsize=14)plt.tight_layout()plt.show()

In [ ]:
# Verteilung wichtiger Features nach reservation_statusfig, axes = plt.subplots(1, 3, figsize=(18, 5))for ax, col in zip(axes, ['lead_time', 'adr', 'total_of_special_requests']):    sns.boxplot(data=df, x=target, y=col, ax=ax, palette=['#2ecc71', '#e74c3c', '#f39c12'])    ax.set_title(f'{col} nach Status', fontsize=12)    ax.set_xlabel('')plt.tight_layout()plt.show()

In [ ]:
# Kategorische Features vs. Targetfig, axes = plt.subplots(1, 3, figsize=(18, 5))for ax, col in zip(axes, ['hotel', 'customer_type', 'deposit_type']):    ct = pd.crosstab(df[col], df[target], normalize='index') * 100    ct.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71', '#e74c3c', '#f39c12'])    ax.set_title(f'{col} vs. {target}', fontsize=12)    ax.set_ylabel('Prozent (%)')    ax.set_xlabel('')    ax.legend(title='Status', fontsize=8)    ax.tick_params(axis='x', rotation=45)plt.tight_layout()plt.show()

## 3 | Feature Engineering & Preprocessing

In [ ]:
# Leakage-Spalten entfernendf = df.drop(columns=hb_cfg['drop_columns'])# Neue Featuresdf['total_stays'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']df['total_guests'] = df['adults'] + df['children'].fillna(0) + df['babies']# Country: Top 10 behalten, Rest = 'Other'top_countries = df['country'].value_counts().head(10).indexdf['country'] = df['country'].where(df['country'].isin(top_countries), 'Other')print(f'Shape nach Feature Engineering: {df.shape}')print(f'Neue Features: total_stays, total_guests')print(f'Country reduziert auf {df["country"].nunique()} Kategorien')

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import LabelEncoder# Target und Features trenneny = df[target]X = df.drop(columns=[target])# Kategorische Spalten encodencat_cols = X.select_dtypes(include='object').columns.tolist()label_encoders = {}for col in cat_cols:    le = LabelEncoder()    X[col] = le.fit_transform(X[col].astype(str))    label_encoders[col] = le# Target encodenle_target = LabelEncoder()y = le_target.fit_transform(y)print(f'Klassen: {list(le_target.classes_)}')print(f'Encoding: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')# Fehlende Werte auffuellenX = X.fillna(0)# Train/Test Split (stratified)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=hb_cfg['test_size'],    random_state=cfg['random_seed'], stratify=y)print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')print(f'Klassenverteilung Train: {np.bincount(y_train)}')print(f'Klassenverteilung Test:  {np.bincount(y_test)}')

## 4 | Baseline-Modell: Logistische Regression

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay# Skalierung fuer LogRegscaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)# Logistische Regression (Multiclass)log_reg = LogisticRegression(max_iter=1000, random_state=cfg['random_seed'],                             class_weight='balanced', multi_class='multinomial')log_reg.fit(X_train_scaled, y_train)y_pred_lr = log_reg.predict(X_test_scaled)print('=== Logistische Regression ===')print(classification_report(y_test, y_pred_lr, target_names=le_target.classes_))fig, ax = plt.subplots(figsize=(8, 6))ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=le_target.classes_,                                         cmap='Blues', ax=ax)ax.set_title('Confusion Matrix: Logistische Regression', fontsize=14)plt.tight_layout()plt.show()

## 5 | Modellvergleich

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.metrics import accuracy_score, f1_score# Modelle definierenmodels = {    'Logistische Regression': (log_reg, X_test_scaled),    'Random Forest': (RandomForestClassifier(        n_estimators=200, random_state=cfg['random_seed'],        class_weight='balanced', n_jobs=-1    ), X_test),    'Gradient Boosting': (GradientBoostingClassifier(        n_estimators=200, random_state=cfg['random_seed'],        max_depth=5, learning_rate=0.1    ), X_test),}# Trainieren und evaluierenresults = []predictions = {}for name, (model, X_eval) in models.items():    X_tr = X_train_scaled if 'Logistisch' in name else X_train    if name != 'Logistische Regression':        model.fit(X_tr, y_train)    y_pred = model.predict(X_eval)    predictions[name] = y_pred    results.append({        'Modell': name,        'Accuracy': accuracy_score(y_test, y_pred),        'Macro F1': f1_score(y_test, y_pred, average='macro'),        'Weighted F1': f1_score(y_test, y_pred, average='weighted'),    })results_df = pd.DataFrame(results).set_index('Modell')display(results_df.round(4))

In [ ]:
# Confusion Matrices im Vergleichfig, axes = plt.subplots(1, 3, figsize=(18, 5))for ax, (name, y_pred) in zip(axes, predictions.items()):    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=le_target.classes_,                                             cmap='Blues', ax=ax, colorbar=False)    ax.set_title(name, fontsize=12)plt.suptitle('Confusion Matrices: Modellvergleich', fontsize=14, y=1.02)plt.tight_layout()plt.show()

In [ ]:
# Metriken-Vergleich als Barplotresults_df.plot(kind='bar', figsize=(10, 5), rot=15, edgecolor='black')plt.title('Modellvergleich: Metriken', fontsize=14)plt.ylabel('Score')plt.ylim(0, 1)plt.legend(loc='lower right')plt.grid(axis='y', linestyle='--', alpha=0.7)plt.tight_layout()plt.show()

## 6 | Hyperparameter-Optimierung

Wir optimieren das beste Modell aus dem Vergleich mit `RandomizedSearchCV`.Das spart Zeit gegenueber `GridSearchCV` bei vielen Parameterkombinationen.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV# Bestes Modell identifizierenbest_model_name = results_df['Macro F1'].idxmax()print(f'Bestes Modell: {best_model_name}')# Hyperparameter-Raeumeparam_distributions = {    'Random Forest': {        'n_estimators': [100, 200, 300, 500],        'max_depth': [10, 20, 30, None],        'min_samples_split': [2, 5, 10],        'min_samples_leaf': [1, 2, 4],        'max_features': ['sqrt', 'log2'],    },    'Gradient Boosting': {        'n_estimators': [100, 200, 300],        'max_depth': [3, 5, 7, 10],        'learning_rate': [0.01, 0.05, 0.1, 0.2],        'min_samples_split': [2, 5, 10],        'subsample': [0.8, 0.9, 1.0],    },}# Falls LogReg bestes Modell ist, optimieren wir stattdessen den besten Tree-Classifierif best_model_name == 'Logistische Regression':    best_model_name = results_df.drop('Logistische Regression')['Macro F1'].idxmax()    print(f'Tree-basiertes Modell fuer Optimierung: {best_model_name}')base_model = models[best_model_name][0].__class__(    random_state=cfg['random_seed'],    **({"class_weight": "balanced", "n_jobs": -1} if "Random Forest" in best_model_name else {}))search = RandomizedSearchCV(    base_model,    param_distributions[best_model_name],    n_iter=20,    cv=3,    scoring='f1_macro',    random_state=cfg['random_seed'],    n_jobs=-1,    verbose=1,)search.fit(X_train, y_train)print(f'\nBeste Parameter: {search.best_params_}')print(f'Bester CV Macro F1: {search.best_score_:.4f}')

In [ ]:
# Optimiertes Modell evaluiereny_pred_opt = search.best_estimator_.predict(X_test)print(f'=== Optimiertes {best_model_name} ===')print(classification_report(y_test, y_pred_opt, target_names=le_target.classes_))# Vergleich vorher/nachherprint(f'Macro F1 vorher:  {results_df.loc[best_model_name, "Macro F1"]:.4f}')print(f'Macro F1 nachher: {f1_score(y_test, y_pred_opt, average="macro"):.4f}')fig, ax = plt.subplots(figsize=(8, 6))ConfusionMatrixDisplay.from_predictions(y_test, y_pred_opt, display_labels=le_target.classes_,                                         cmap='Blues', ax=ax)ax.set_title(f'Confusion Matrix: Optimiertes {best_model_name}', fontsize=14)plt.tight_layout()plt.show()

## 7 | Feature Importance

In [ ]:
# Feature Importance des optimierten Modellsimportances = search.best_estimator_.feature_importances_feat_imp = pd.DataFrame({    'Feature': X_train.columns,    'Importance': importances}).sort_values('Importance', ascending=True).tail(15)plt.figure(figsize=(10, 8))plt.barh(feat_imp['Feature'], feat_imp['Importance'], color='steelblue', edgecolor='black')plt.title(f'Top 15 Feature Importances ({best_model_name})', fontsize=14)plt.xlabel('Importance')plt.grid(axis='x', linestyle='--', alpha=0.7)plt.tight_layout()plt.show()

## 8 | Fazit**Zusammenfassung:**- Der Hotel-Bookings-Datensatz wurde fuer eine Multiclass-Klassifikation (Check-Out / Canceled / No-Show) aufbereitet.- Data Leakage wurde durch Entfernung von `is_canceled` und `reservation_status_date` verhindert.- Drei Modelle (Logistische Regression, Random Forest, Gradient Boosting) wurden verglichen.- Das beste Modell wurde per RandomizedSearchCV optimiert.**Herausforderungen:**- **Class Imbalance:** No-Show ist eine sehr seltene Klasse und damit schwer vorherzusagen.- `class_weight='balanced'` hilft, reicht aber bei extremer Imbalance nicht immer aus.- Fuer Produktionsszenarien waeren Oversampling (SMOTE) oder angepasste Schwellenwerte sinnvoll.